<a href="https://www.kaggle.com/code/asivakumarnair/diabetic-retinopathy-imagenet?scriptVersionId=344139475" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [2]:
# ===== STAGE 12 SESSION: FULL REBUILD + CROSS-SOURCE 3x3 MATRIX (chest's schema + QWK) =====
!pip install -q tensorflow==2.19.0

import os
os.environ['TF_USE_LEGACY_KERAS'] = '1'

import random
import numpy as np
import pandas as pd
import gc
import tensorflow as tf

SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
os.environ['TF_DETERMINISTIC_OPS'] = '1'
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
print(f"Seed {SEED} set, TF {tf.__version__}, tf.keras module: {tf.keras.__name__}")

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.efficientnet import preprocess_input as eff_pre
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input as mob_pre
from tensorflow.keras.applications.resnet50 import preprocess_input as res_pre
from tensorflow.keras.models import load_model
from sklearn.model_selection import train_test_split
from sklearn.metrics import (roc_auc_score, accuracy_score, f1_score,
                              confusion_matrix, recall_score, cohen_kappa_score)

# ---------- CONFIG ----------
APTOS_CSV     = '/kaggle/input/competitions/aptos2019-blindness-detection/train.csv'
APTOS_IMG     = '/kaggle/input/competitions/aptos2019-blindness-detection/train_images'
EYEPACS_CSV   = '/kaggle/input/datasets/benjaminwarner/resized-2015-2019-blindness-detection-images/labels/trainLabels15.csv'
EYEPACS_IMG   = '/kaggle/input/datasets/benjaminwarner/resized-2015-2019-blindness-detection-images/resized train 15'
MESSIDOR_CSV  = '/kaggle/input/datasets/mariaherrerot/messidor2preprocess/messidor_data.csv'
MESSIDOR_IMG  = '/kaggle/input/datasets/mariaherrerot/messidor2preprocess/messidor-2/messidor-2/preprocess'

SS_DIR = '/kaggle/input/datasets/asivakumarnair/drstage11singlesource/'

GRADES      = ['0','1','2','3','4']
NUM_CLASSES = 5
IMG_SIZE, BATCH_SIZE = 224, 32
SUBSAMPLE_SEED = 42
EYEPACS_TARGET = 3662

# ---------- DATA REBUILD, all three sources ----------
aptos = pd.read_csv(APTOS_CSV)
aptos['grade']      = aptos['diagnosis'].astype(int).astype(str)
aptos['image_path'] = APTOS_IMG + '/' + aptos['id_code'].astype(str) + '.png'
aptos['source']     = 'aptos'
aptos['patient_id'] = None

eyepacs = pd.read_csv(EYEPACS_CSV)
eyepacs['grade']      = eyepacs['level'].astype(int).astype(str)
eyepacs['image_path'] = EYEPACS_IMG + '/' + eyepacs['image'].astype(str) + '.jpg'
eyepacs['source']     = 'eyepacs'
eyepacs['patient_id'] = eyepacs['image'].str.extract(r'^(\d+)_')
assert eyepacs['patient_id'].isna().sum() == 0, "EyePACS patient_id extraction failed"

messidor = pd.read_csv(MESSIDOR_CSV)
messidor['grade']      = messidor['diagnosis'].astype(int).astype(str)
messidor['image_path'] = MESSIDOR_IMG + '/' + messidor['id_code'].astype(str)
messidor['source']     = 'messidor'
messidor['patient_id'] = None

def subsample_eyepacs(df, target_n=EYEPACS_TARGET, seed=SUBSAMPLE_SEED):
    pg = df.groupby('patient_id')['grade'].max().reset_index()
    frac = target_n / len(df)
    keep, _ = train_test_split(pg, train_size=frac, stratify=pg['grade'], random_state=seed)
    return df[df['patient_id'].isin(keep['patient_id'])].reset_index(drop=True)

eyepacs_s = subsample_eyepacs(eyepacs)

def safe_split(df, label_col, test_size, rs, tag=""):
    try:
        return train_test_split(df, test_size=test_size, stratify=df[label_col], random_state=rs)
    except ValueError as e:
        print(f"WARNING [{tag}]: stratified split failed, falling back to unstratified.")
        return train_test_split(df, test_size=test_size, random_state=rs)

def split_patient_level(df, rs=SEED, tag=""):
    pg = df.groupby('patient_id')['grade'].max().reset_index()
    p_tr, p_tmp = safe_split(pg, 'grade', 0.30, rs, tag=f"{tag} first")
    p_va, p_te  = safe_split(p_tmp, 'grade', 0.50, rs, tag=f"{tag} second")
    pick = lambda ids: df[df['patient_id'].isin(ids['patient_id'])]
    tr, va, te = pick(p_tr), pick(p_va), pick(p_te)
    s_tr, s_va, s_te = set(p_tr['patient_id']), set(p_va['patient_id']), set(p_te['patient_id'])
    assert s_tr.isdisjoint(s_va) and s_tr.isdisjoint(s_te) and s_va.isdisjoint(s_te), f"{tag} PATIENT LEAKAGE"
    print(f"{tag} patient-leakage check: PASS")
    return tr, va, te

def split_image_level(df, rs=SEED, tag=""):
    tr, tmp = safe_split(df, 'grade', 0.30, rs, tag=f"{tag} first")
    va, te  = safe_split(tmp, 'grade', 0.50, rs, tag=f"{tag} second")
    return tr, va, te

def split_messidor_mixed(df, rs=SEED, tag="Messidor"):
    is_im = ~df['image_path'].str.contains(r'\d{8}_\d+_\d+_PP\.png$', regex=True)
    im_df = df[is_im].copy()
    im_df['im_num'] = im_df['image_path'].str.extract(r'IM(\d+)\.JPG$').astype(int)
    im_df = im_df.sort_values('im_num').reset_index(drop=True)
    im_df['patient_id'] = 'messidor_pair_' + (im_df.index // 2).astype(str)
    pg = im_df.groupby('patient_id')['grade'].max().reset_index()
    p_tr, p_tmp = safe_split(pg, 'grade', 0.30, rs, tag=f"{tag} IM first")
    p_va, p_te  = safe_split(p_tmp, 'grade', 0.50, rs, tag=f"{tag} IM second")
    pick = lambda ids: im_df[im_df['patient_id'].isin(ids['patient_id'])]
    im_tr, im_va, im_te = pick(p_tr), pick(p_va), pick(p_te)
    s_tr, s_va, s_te = set(p_tr['patient_id']), set(p_va['patient_id']), set(p_te['patient_id'])
    assert s_tr.isdisjoint(s_va) and s_tr.isdisjoint(s_te) and s_va.isdisjoint(s_te), "MESSIDOR IM PATIENT LEAKAGE"
    print(f"{tag} IM-style patient-leakage check: PASS")
    date_df = df[~is_im]
    d_tr, d_va, d_te = split_image_level(date_df, rs, tag=f"{tag} date-style")
    cat = lambda a, b: pd.concat([a.drop(columns=['im_num']), b], ignore_index=True)
    return cat(im_tr, d_tr), cat(im_va, d_va), cat(im_te, d_te)

a_tr, a_va, a_te = split_image_level(aptos, tag="APTOS")
e_tr, e_va, e_te = split_patient_level(eyepacs_s, tag="EyePACS")
m_tr, m_va, m_te = split_messidor_mixed(messidor)

test_sets = {'aptos': a_te, 'eyepacs': e_te, 'messidor': m_te}
for src, d in test_sets.items():
    print(f"{src} test set: {len(d):,} images")
print("Expect: aptos 550 | eyepacs 550 | messidor 263")

# ---------- MODEL PATHS, explicit, not a glob (phase1 files must not leak in) ----------
ARCH_SPECS   = {'custom': None, 'eff': eff_pre, 'mob': mob_pre, 'res': res_pre}
ARCH_DISPLAY = {'custom':'Custom CNN','eff':'EfficientNetB0','mob':'MobileNetV2','res':'ResNet50'}
SOURCES      = ['aptos', 'eyepacs', 'messidor']

MODEL_PATHS_SS = {}
for arch in ARCH_SPECS:
    for src in SOURCES:
        p = f'{SS_DIR}ss_{arch}_{src}_dr.keras'
        assert os.path.exists(p), f"MISSING checkpoint: {p}"
        MODEL_PATHS_SS[(arch, src)] = p
        print(f"  {arch:7} {src:9} -> FOUND")

# ---------- GATE: Stage 11 locked ACCURACY values ----------
# Accuracy only, deliberately. Stage 11's AUC came from Keras's built-in tf.keras.metrics.AUC
# during model.evaluate(), while this stage uses sklearn's macro-averaged roc_auc_score.
# Those are different computations and will not match, exactly as Stage 8 already showed for
# the pooled models (Custom CNN: 0.836 training-time vs 0.690 post-hoc). Accuracy uses the
# same argmax computation in both places, so it is the valid reproduction check.
STAGE11_ACC_GATE = {
    ('custom','aptos'):    0.700000, ('custom','eyepacs'):  0.158182, ('custom','messidor'):  0.581749,
    ('eff','aptos'):       0.672727, ('eff','eyepacs'):     0.630909, ('eff','messidor'):     0.608365,
    ('mob','aptos'):       0.632727, ('mob','eyepacs'):     0.558182, ('mob','messidor'):     0.608365,
    ('res','aptos'):       0.772727, ('res','eyepacs'):     0.550909, ('res','messidor'):     0.498099,
}

def make_eval_gen(df, preprocess_fn):
    idg = (ImageDataGenerator(rescale=1./255) if preprocess_fn is None
           else ImageDataGenerator(preprocessing_function=preprocess_fn))
    return idg.flow_from_dataframe(df, x_col='image_path', y_col='grade',
                                    target_size=(IMG_SIZE,IMG_SIZE), batch_size=BATCH_SIZE,
                                    class_mode='categorical', classes=GRADES,
                                    color_mode='rgb', shuffle=False)

def macro_specificity(y_true, y_pred, n_classes=NUM_CLASSES):
    cm = confusion_matrix(y_true, y_pred, labels=range(n_classes))
    total = cm.sum(); specs = []
    for i in range(n_classes):
        tp = cm[i,i]; fn = cm[i,:].sum()-tp; fp = cm[:,i].sum()-tp
        tn = total-tp-fn-fp
        specs.append(tn/(tn+fp) if (tn+fp) > 0 else np.nan)
    return np.nanmean(specs)

# ---------- RUN: 12 models x 3 test sets = 36 cells ----------
rows = []
for arch, preproc in ARCH_SPECS.items():
    for train_src in SOURCES:
        path = MODEL_PATHS_SS[(arch, train_src)]
        print(f"\nLoading {ARCH_DISPLAY[arch]} trained-on-{train_src}")
        m = load_model(path)
        for test_src in SOURCES:
            gen = make_eval_gen(test_sets[test_src], preproc)
            y_true = np.asarray(gen.classes)
            y_prob = m.predict(gen, verbose=0)
            y_pred = np.argmax(y_prob, axis=1)

            try:
                auc = roc_auc_score(np.eye(NUM_CLASSES)[y_true], y_prob,
                                     average='macro', multi_class='ovr')
            except ValueError as e:
                auc = np.nan
                print(f"  WARNING: macro_auc failed for {arch}/{train_src}->{test_src}: {e}")

            setting = 'within' if train_src == test_src else 'cross'
            row = {
                'arch': ARCH_DISPLAY[arch], 'train_source': train_src, 'test_source': test_src,
                'setting': setting, 'n': len(y_true),
                'macro_auc': auc,
                'macro_f1': f1_score(y_true, y_pred, average='macro'),
                'accuracy': accuracy_score(y_true, y_pred),
                'macro_sensitivity': recall_score(y_true, y_pred, average='macro'),
                'macro_specificity': macro_specificity(y_true, y_pred),
                'qwk': cohen_kappa_score(y_true, y_pred, weights='quadratic'),
            }
            rows.append(row)
            print(f"  -> {test_src:9} [{setting:6}] n={row['n']:4} AUC={auc:.4f} "
                  f"QWK={row['qwk']:.4f} Acc={row['accuracy']:.4f} F1={row['macro_f1']:.4f}")

            np.savez(f'/kaggle/working/s12_preds_{arch}_{train_src}_on_{test_src}.npz',
                     y_true=y_true, y_pred=y_pred, y_prob=y_prob)
        del m; gc.collect(); tf.keras.backend.clear_session()

s12 = pd.DataFrame(rows)
s12.to_csv('/kaggle/working/dr_stage12_crosssource.csv', index=False)

# ---------- GATE CHECK ----------
print("\n===== GATE: within-source accuracy reproduction vs Stage 11 =====")
ok = True
for (arch, src), exp_acc in STAGE11_ACC_GATE.items():
    r = s12[(s12.arch==ARCH_DISPLAY[arch]) & (s12.train_source==src) & (s12.test_source==src)].iloc[0]
    d = abs(r.accuracy - exp_acc)
    flag = "OK      " if d < 1e-3 else "MISMATCH"
    if d >= 1e-3: ok = False
    print(f"{flag} {ARCH_DISPLAY[arch]:15} {src:9} acc {r.accuracy:.6f} vs Stage11 {exp_acc:.6f}  diff {d:.6f}")
print("\nALL WITHIN-SOURCE CELLS REPRODUCE" if ok
      else "\nGATE FAILED, stop and investigate before using any cross-source number")

# ---------- PRIMARY TABLE: macro-AUC ----------
print("\n===== CROSS-SOURCE MATRIX, macro-AUC (PRIMARY) =====")
print(s12.pivot_table(index=['arch','train_source'], columns='test_source',
                       values='macro_auc').round(4).to_string())

print("\n===== CROSS-SOURCE MATRIX, QWK (ordinal, DR-specific) =====")
print(s12.pivot_table(index=['arch','train_source'], columns='test_source',
                       values='qwk').round(4).to_string())

# ---------- OFF-DIAGONAL AUC DROP, matching chest's summary ----------
print("\n===== OFF-DIAGONAL (cross) AUC DROP PER ARCHITECTURE =====")
print("(chest reference: Custom CNN mean 0.2153 | eff 0.1206 | mob 0.0869 | res 0.1056)")
drop_rows = []
for arch_d in ARCH_DISPLAY.values():
    sub = s12[s12.arch == arch_d]
    within = sub[sub.setting=='within'].set_index('train_source')['macro_auc']
    cross  = sub[sub.setting=='cross']
    drops = [within[r.train_source] - r.macro_auc for r in cross.itertuples()]
    drop_rows.append({'arch': arch_d, 'mean_auc_drop': np.mean(drops),
                      'median_auc_drop': np.median(drops),
                      'min_cross_auc': cross.macro_auc.min(),
                      'max_cross_auc': cross.macro_auc.max()})
    print(f"{arch_d:15} mean drop = {np.mean(drops):.4f} | median drop = {np.median(drops):.4f} | "
          f"min cross AUC = {cross.macro_auc.min():.4f} | max cross AUC = {cross.macro_auc.max():.4f}")
pd.DataFrame(drop_rows).to_csv('/kaggle/working/dr_stage12_auc_drop_summary.csv', index=False)

print("\n===== FULL TABLE =====")
print(s12.round(4).to_string(index=False))
print("\nSaved: dr_stage12_crosssource.csv, dr_stage12_auc_drop_summary.csv, s12_preds_*.npz (36 files)")

Seed 42 set, TF 2.19.0, tf.keras module: tensorflow.keras
EyePACS patient-leakage check: PASS
WARNING [Messidor IM second]: stratified split failed, falling back to unstratified.
Messidor IM-style patient-leakage check: PASS
aptos test set: 550 images
eyepacs test set: 550 images
messidor test set: 263 images
Expect: aptos 550 | eyepacs 550 | messidor 263
  custom  aptos     -> FOUND
  custom  eyepacs   -> FOUND
  custom  messidor  -> FOUND
  eff     aptos     -> FOUND
  eff     eyepacs   -> FOUND
  eff     messidor  -> FOUND
  mob     aptos     -> FOUND
  mob     eyepacs   -> FOUND
  mob     messidor  -> FOUND
  res     aptos     -> FOUND
  res     eyepacs   -> FOUND
  res     messidor  -> FOUND

Loading Custom CNN trained-on-aptos
Found 550 validated image filenames belonging to 5 classes.
  -> aptos     [within] n= 550 AUC=0.8427 QWK=0.6600 Acc=0.7000 F1=0.4575
Found 550 validated image filenames belonging to 5 classes.
  -> eyepacs   [cross ] n= 550 AUC=0.4795 QWK=0.0764 Acc=0.7091